In [393]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import time
import random
import itertools

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Number of GPUs:", torch.cuda.device_count())

PyTorch version: 2.13.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1


In [394]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: cuda:0


In [395]:
if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)}"
        )

        print(
            f"Memory: "
            f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB"
        )

else:

    print("Running on CPU")

GPU 0: NVIDIA GeForce GTX 1650 Ti
Memory: 4.00 GB


In [396]:
if not dist.is_initialized():

    dist.init_process_group(
        backend="gloo",
        init_method="tcp://127.0.0.1:29501",
        rank=0,
        world_size=1
    )

print("Distributed initialized")
print("Backend:", dist.get_backend())
print("Rank:", dist.get_rank())
print("World size:", dist.get_world_size())

Distributed initialized
Backend: gloo
Rank: 0
World size: 1


In [397]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Linear(
            64 * 8 * 8,
            num_classes
        )

    def forward(self, x):

        x = self.conv1(x)
        x = F.relu(x)

        x = self.pool(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x

In [398]:
model = SimpleCNN(num_classes = 10).to(device)


In [399]:
total_parameters = sum(p.numel() for p in model.parameters())

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Total parameters: 60362
Trainable parameters: 60362


In [400]:
image_batch = torch.randn(64,3,32,32,device=device)

print("Image batch shape:", image_batch.shape)
print("Device:", image_batch.device)

Image batch shape: torch.Size([64, 3, 32, 32])
Device: cuda:0


In [401]:
model.eval()

with torch.no_grad():

    model_output = model(image_batch)

print("Input :", image_batch.shape)
print("Output:", model_output.shape)

Input : torch.Size([64, 3, 32, 32])
Output: torch.Size([64, 10])


In [402]:
num_partitions = 4

sample_partitions = torch.chunk(
    image_batch,
    chunks=num_partitions,
    dim=0
)

for i, partition in enumerate(sample_partitions):

    print(
        f"Partition {i}:",
        partition.shape
    )

Partition 0: torch.Size([16, 3, 32, 32])
Partition 1: torch.Size([16, 3, 32, 32])
Partition 2: torch.Size([16, 3, 32, 32])
Partition 3: torch.Size([16, 3, 32, 32])


In [403]:
model.eval()

sample_outputs = []

with torch.no_grad():

    for i, partition in enumerate(sample_partitions):

        partition = partition.to(device)

        output = model(partition)

        sample_outputs.append(output)

        print(
            f"Partition {i}:",
            partition.shape,
            "→",
            output.shape
        )

Partition 0: torch.Size([16, 3, 32, 32]) → torch.Size([16, 10])
Partition 1: torch.Size([16, 3, 32, 32]) → torch.Size([16, 10])
Partition 2: torch.Size([16, 3, 32, 32]) → torch.Size([16, 10])
Partition 3: torch.Size([16, 3, 32, 32]) → torch.Size([16, 10])


In [404]:
sample_parallel_output = torch.cat(
    sample_outputs,
    dim=0
)

print(
    "Combined output:",
    sample_parallel_output.shape
)

Combined output: torch.Size([64, 10])


In [405]:
class Stage1(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

    def forward(self, input_tensor):

        output = self.conv1(input_tensor)

        output = F.relu(output)

        return output

In [406]:
class Stage2(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Linear(
            64 * 16 * 16,
            10
        )

    def forward(self, input_tensor):

        output = self.conv2(input_tensor)

        output = F.relu(output)

        output = self.pool(output)

        output = torch.flatten(
            output,
            start_dim=1
        )

        output = self.fc(output)

        return output

In [407]:
stage1 = Stage1().to(device)
stage2 = Stage2().to(device)

print(
    "Stage 1 device:",
    next(stage1.parameters()).device
)

print(
    "Stage 2 device:",
    next(stage2.parameters()).device
)

Stage 1 device: cuda:0
Stage 2 device: cuda:0


In [408]:
operator_input = torch.randn(
    8,
    3,
    32,
    32,
    device=device
)

print(
    "Operator input:",
    operator_input.shape
)

print(
    "Device:",
    operator_input.device
)

Operator input: torch.Size([8, 3, 32, 32])
Device: cuda:0


In [409]:
stage1_output = stage1(
    operator_input
)

print(
    "Stage 1 output:",
    stage1_output.shape
)

print(
    "Device:",
    stage1_output.device
)

Stage 1 output: torch.Size([8, 32, 32, 32])
Device: cuda:0


In [410]:
stage2_output = stage2(
    stage1_output
)

print(
    "Stage 2 output:",
    stage2_output.shape
)

print(
    "Device:",
    stage2_output.device
)

Stage 2 output: torch.Size([8, 10])
Device: cuda:0


In [411]:
single_image = torch.randn(
    1,
    3,
    32,
    32
)

print(
    "Image:",
    single_image.shape
)

Image: torch.Size([1, 3, 32, 32])


In [412]:
height_parts = torch.chunk(
    single_image,
    chunks=2,
    dim=2
)

for i, part in enumerate(height_parts):

    print(
        f"Height partition {i}:",
        part.shape
    )

Height partition 0: torch.Size([1, 3, 16, 32])
Height partition 1: torch.Size([1, 3, 16, 32])


In [413]:
width_parts = torch.chunk(
    single_image,
    chunks=2,
    dim=3
)

for i, part in enumerate(width_parts):

    print(
        f"Width partition {i}:",
        part.shape
    )

Width partition 0: torch.Size([1, 3, 32, 16])
Width partition 1: torch.Size([1, 3, 32, 16])


In [414]:
channel_parts = torch.chunk(
    single_image,
    chunks=3,
    dim=1
)

for i, part in enumerate(channel_parts):

    print(
        f"Channel partition {i}:",
        part.shape
    )

Channel partition 0: torch.Size([1, 1, 32, 32])
Channel partition 1: torch.Size([1, 1, 32, 32])
Channel partition 2: torch.Size([1, 1, 32, 32])


In [415]:
print("""
Original tensor:

[Batch, Channels, Height, Width]

[1, 3, 32, 32]


Possible attribute partitions:

Height:
[1, 3, 16, 32] + [1, 3, 16, 32]

Width:
[1, 3, 32, 16] + [1, 3, 32, 16]

Channels:
[1, 1, 32, 32] × 3
""")


Original tensor:

[Batch, Channels, Height, Width]

[1, 3, 32, 32]


Possible attribute partitions:

Height:
[1, 3, 16, 32] + [1, 3, 16, 32]

Width:
[1, 3, 32, 16] + [1, 3, 32, 16]

Channels:
[1, 1, 32, 32] × 3



In [416]:
large_linear = nn.Linear(4096,4096)

print("Weight shape:",large_linear.weight.shape)

print("Bias shape:",large_linear.bias.shape)

Weight shape: torch.Size([4096, 4096])
Bias shape: torch.Size([4096])


In [417]:
linear_parameters = sum(
    parameter.numel()
    for parameter in large_linear.parameters()
)

print(
    "Total parameters:",
    linear_parameters
)

Total parameters: 16781312


In [418]:
weight_matrix = large_linear.weight

parameter_shards = torch.chunk(
    weight_matrix,
    chunks=4,
    dim=0
)

for i, shard in enumerate(parameter_shards):

    print(
        f"Shard {i}:",
        shard.shape
    )

Shard 0: torch.Size([1024, 4096])
Shard 1: torch.Size([1024, 4096])
Shard 2: torch.Size([1024, 4096])
Shard 3: torch.Size([1024, 4096])


In [419]:
for i, shard in enumerate(parameter_shards):

    print(
        f"Shard {i} parameters:",
        shard.numel()
    )

Shard 0 parameters: 4194304
Shard 1 parameters: 4194304
Shard 2 parameters: 4194304
Shard 3 parameters: 4194304


In [420]:
soap = {
    "S": "Sample",
    "O": "Operator",
    "A": "Attribute",
    "P": "Parameter"
}

for key, meaning in soap.items():

    print(
        f"{key} = {meaning}"
    )

S = Sample
O = Operator
A = Attribute
P = Parameter


In [421]:
soap_dimensions = [
    "S",
    "O",
    "A",
    "P"
]

all_strategies = []

for r in range(1, len(soap_dimensions) + 1):

    combinations = itertools.combinations(
        soap_dimensions,
        r
    )

    for combination in combinations:

        strategy = " + ".join(
            combination
        )

        all_strategies.append(
            strategy
        )

for i, strategy in enumerate(all_strategies):

    print(
        f"{i+1:2d}. {strategy}"
    )

 1. S
 2. O
 3. A
 4. P
 5. S + O
 6. S + A
 7. S + P
 8. O + A
 9. O + P
10. A + P
11. S + O + A
12. S + O + P
13. S + A + P
14. O + A + P
15. S + O + A + P


In [422]:
print(
    "Number of non-empty SOAP combinations:",
    len(all_strategies)
)

Number of non-empty SOAP combinations: 15


In [423]:
def estimate_cost(
    compute_cost,
    communication_cost,
    num_devices
):

    computation_time = (
        compute_cost / num_devices
    )

    communication_time = (
        communication_cost *
        max(num_devices - 1, 0)
    )

    total_cost = (
        computation_time +
        communication_time
    )

    return total_cost

In [424]:
compute_cost = 100
communication_cost = 5

for number_of_devices in [1, 2, 4, 8]:

    cost = estimate_cost(
        compute_cost=compute_cost,
        communication_cost=communication_cost,
        num_devices=number_of_devices
    )

    print(
        f"{number_of_devices} devices "
        f"→ cost = {cost:.2f}"
    )

1 devices → cost = 100.00
2 devices → cost = 55.00
4 devices → cost = 40.00
8 devices → cost = 47.50
